# Bayes Notebook
**Bayes_HEP: Bayes_Main**

<details>
<summary>
What is this notebook?
</summary>

This notebook runs a notebook version of Bayes_Main, which is the **second half** of the Bayesian tuning pipeline. It uses the simulation outputs from `Rivet_Notebook.ipynb` to train a fast **Gaussian Process emulator**, then runs **Bayesian calibration** (MCMC or nested sampling) to find the Pythia8 parameter values that best match experimental data.

> **New to this?** A *Gaussian Process* is a probabilistic model that learns to mimic expensive simulations with a cheap surrogate. *Bayesian calibration* uses Bayes' theorem to turn that surrogate into a probability distribution over the parameters — telling us not just the best-fit values, but how uncertain we are about them.

</details>

<details>
<summary>
Why do we need this?
</summary>

Running Pythia8 for every possible parameter combination is computationally infeasible. The GP emulator trained in this notebook acts as a fast stand-in for Pythia8, allowing us to evaluate millions of parameter combinations during calibration in seconds rather than hours. The result is a **posterior distribution** — a principled, uncertainty-aware estimate of the best-fit parameters.

</details>

<details>
<summary>
What does this notebook do?
</summary>

1. **Configuration** — set your project paths and choose which pipeline stages to run
2. **Directory setup** — create the output directory for plots, emulators, and calibration results
3. **Design points** — load and merge the Pythia8 design point files from `Rivet_Notebook.ipynb`
4. **Load data and predictions** — read experimental measurements and Pythia8 predictions, split into train/validation sets
5. **Emulator training** — fit a Gaussian Process to the training data and validate on held-out points
6. **Closure test** — validate the calibration pipeline against a held-out design point with known truth
7. **Bayesian calibration** — run MCMC or nested sampling to build the posterior distribution
8. **Results** — visualize trace plots, posterior predictive bands, and parameter constraints

</details>

## Workflow

Run `Rivet_Notebook.ipynb` **before** this notebook. The `input/Data/` and `input/Prediction/` files produced there feed directly into the Bayesian calibration pipeline.

---

## Companion Slides

<details>
<summary>
The <code>Bayes_HEP_Project.pptx</code> slide deck provides background for each section of this notebook. Refer to these slides as you work through the cells:
</summary>

| Notebook section | Relevant slides |
|-----------------|----------------|
| Full workflow overview | **Slides 3–6** — Bayesian inference diagram, pipeline steps |
| Setup & tools | **Slides 11–15** — VS Code, Docker, Apptainer, ISAAC HPC, remote access |
| Physics background | **Slides 16–20** — QCD, QGP, RHIC/LHC, recommended papers |
| Design points | **Slides 26–27** — Latin Hypercube Sampling, DETMAX algorithm |
| Gaussian Process emulator | **Slides 36–40** — GP regression, kernel functions, LOO validation |
| Bayesian calibration | **Slides 41–47** — Bayes' theorem, MCMC, emcee, nested sampling |
| Results | **Slides 48–52** — posterior corner plots, predictive bands, MAP estimates |

</details>

### Step 1 — Setup (run once)

<details>
<summary>From the <code>Bayes_HEP</code> directory on ISAAC, run:</summary>

```bash
bash New_Project/drivers/notebook/utilities/setup.sh <username>
```

Replace `<username>` with your ISAAC username (e.g. `cbaillar`).

</details>

### Step 2 — Start Jupyter

<details>
<summary>In your ISAAC terminal, run:</summary>

```bash
jupyter notebook --no-browser --port=8888 --ip=0.0.0.0
```

Copy the full URL printed:
```
http://127.0.0.1:8888/?token=abc123...
```

</details>

### Step 3 — Open the notebook in VSCode

<details>
<summary>Connect to Jupyter server</summary>

1. Open the notebook via Remote-SSH in VSCode
2. Click the kernel selector in the top right corner
3. Select **Existing Jupyter Server**
4. Paste the token URL from Step 2
5. Select kernel
- For physics/analysis cells → select **Bayes HEP (Apptainer)**
- For SLURM job submission cells → select **Python 3**

> **Note:** The Jupyter server must be running before opening the notebook in VSCode.

</details>

## 1. Configuration

**This is the main cell you need to edit for your own project.** Set your paths and choose which pipeline stages to run.

### Paths

<details>
<summary>
Path details:
</summary>

| Variable | What to set |
|----------|-------------|
| `username` | Your ISAAC HPC username (the part after `/UTK0244/`) |
| `work_dir` | Base directory on ISAAC where your Bayes_HEP project lives |
| `main_dir` | The project directory (auto-built from `work_dir`) |
| `input` | Name of the input subdirectory inside the project directory |
| `output` | Name of the output subdirectory inside the project directory |

</details>

### Pipeline Toggles

<details>
<summary>
Each boolean turns a pipeline stage on (<code>True</code>) or off (<code>False</code>). For a clean first run, set all to <code>True</code> and run top to bottom. On subsequent runs you can skip stages you've already completed.
</summary>

#

| Flag | What it does |
|------|--------------|
| `clear_output` | Wipes the `output/` directory before running — ensures no stale plots or emulator files from a previous run. Set `False` to preserve existing outputs. |
| `Train_Surmise` | `True` → train the GP emulator interactively in-notebook. |
| `Load_Surmise` | `True` → load a previously saved emulator from `output/emulator/`. Use when design points haven't changed. Skips retraining. |
| `Submit_job_train` | Submit a SLURM batch job to train the emulator on Isaac HPC. Recommended for large design matrices (200+ design points). |
| `Submit_job_calibrate` | Submit a SLURM batch job to run calibration on Isaac HPC. Recommended for production runs. |
| `Run_Calibration` | Run calibration interactively in-notebook. Use for small tests only — not suitable for large sample counts. |
| `show_plots` | Display result plots inline in the notebook after calibration. |

</details>

### Sampler Settings

<details>
<summary>
Key parameters that control how the posterior is sampled:
</summary>

#

| Variable | Description |
|----------|-------------|
| `SAMPLERS` | List of samplers to use: `"emcee"` (MCMC) and/or `"dynesty"` (nested sampling) |
| `nwalkers` | Number of MCMC walkers for emcee. More walkers → better mixing, slower runtime. |
| `Samples` | Number of MCMC steps per walker. More steps → better-converged posterior. |
| `nburn` | Burn-in steps to discard from the chain start. |
| `npool` | Number of parallel processes for likelihood evaluation. |

</details>

### Required Input Files

Before running, confirm these files exist in `input/` — all are produced by `Rivet_Notebook.ipynb`:

<details>
<summary><b>📁 input/Design/Design__Rivet__*.dat</b> — design point files</summary>

One or more numbered files (e.g., `Design__Rivet__1.dat`, `Design__Rivet__2.dat`) containing the Pythia8 parameter combinations sampled via LHS. Generated by Section 3 of `Rivet_Notebook.ipynb`.

This notebook merges all `Design__Rivet__*.dat` files into a single `Design__Rivet__Merged.dat` before training. If only one file exists, it is used directly.

</details>

<details>
<summary><b>📁 input/Data/Data__*.dat</b> — experimental data files</summary>

One file per observable containing the real detector measurements (x-values, y-values, statistical and systematic uncertainties). Generated by Section 8 of `Rivet_Notebook.ipynb`.

File naming follows: `Data__{Energy}__{System}__{Analysis}__{Histogram}`

</details>

<details>
<summary><b>📁 input/Prediction/Prediction__*.dat</b> — Pythia8 prediction files</summary>

One file per observable per design point batch containing the Pythia8-simulated values at each design point. Generated by Section 8 of `Rivet_Notebook.ipynb`.

File naming follows: `Prediction__{model}__{Energy}__{System}__{Analysis}__{Histogram}__DG_{N}`

</details>

<details>
<summary><b>📄 input/Rivet/parameter_prior_list.dat</b> — parameter names and prior ranges</summary>

Defines the Pythia8 parameters being tuned and the prior ranges used during design point generation. This is the same file configured in `Rivet_Notebook.ipynb` — no changes needed here unless you are modifying the parameter set.

```
# Parameter pT0Ref ecmPow coreRadius coreFraction CRrange
# - Parameter pT0Ref:       Linear [0.5, 2.5]
# - Parameter ecmPow:       Linear [0.0, 0.25]
# - Parameter coreRadius:   Linear [0.1, 1.0]
# - Parameter coreFraction: Linear [0.0, 1.0]
# - Parameter CRrange:      Linear [1.0, 9.0]
```

</details>

In [1]:
# Run for both kernels

username    = 'cbaillar'
project     = 'New_Project'
input       = 'input'
output      = 'output'

work_dir = f'/lustre/isaac24/proj/UTK0244/{username}/Bayes_HEP'
main_dir   = f"{work_dir}/{project}"
input_dir = f"{main_dir}/{input}" 
output_dir = f"{main_dir}/{output}"

hpc_dir=f"{main_dir}/Batch_Jobs/HPC/notebook"
CONTAINER=f"{work_dir}/bayes_hep.sif"
BIND_PATH=f"{work_dir}:/workdir"

QOS         = 'campus'

QOS_WALLTIMES = {
    'short'  : '2:55:00',
    'campus' : '23:55:00',
    'long'   : '143:55:00'
}

WALLTIME = QOS_WALLTIMES[QOS]

error_path  = f"/lustre/isaac24/scratch/{username}/jobs/error/job.e%J"
output_path = f"/lustre/isaac24/scratch/{username}/jobs/output/job.o%J"

In [2]:
#Run for both kernels
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

clear_output = False

model = 'pythia8'
Coll_System = ['all'] #['all', 'pp_200'] 

######## Design Points
train_size      = 50
validation_size = 10

######## Preprocessing
PCA   = False
scalers = None   #scikit-learn scaling not yet supported -- leave as None (Surmise scales internally)

######## Emulators
Submit_job_train = False     #train emulator(s) by submitting slurm job (recommended for >50 DP)
Train_Surmise    = False       #train emulator(s) interactively
Load_Surmise     = False       #load emulator(s) interactively


######## Calibration
Submit_job_closure  = False
Run_Closure         = False   #run closure calibration interactively (can be slow -- prefer Submit_job_closure for production)
Run_Closure_Plots   = False   #generate closure plots (run after calibration, interactive or SLURM, has finished)
closure_index       = 25

Submit_job_calibrate = False #run calibration by submitting slurm job (recommended)
Run_Calibration      = False      

SAMPLERS  = ["dynesty"] #["emcee", "dynesty"]
npool     = 10
cov_mode  = 'full'      #full or diag

#emcee settings
Samples      = 1000
thin_samples = 1
burn_frac    = 0.10

#dynesty setting
dlog     = 0.01

######## Results
Submit_job_results = True
Run_Results        = False
size               = 200    #number of samples used
show_plots         = False

######## MAP
Submit_Job_MAP  = False
MAP_merge       = False
MAP_write       = False
RUN_PT_HAT_BINS = False
nevents         = 2000000   #per job

In [3]:
# Run for both kernels

import os
import shutil
import matplotlib.pyplot as plt
import glob
import numpy as np
import sys

def get_kernel():
    return 'apptainer' if 'apptainer' in sys.executable.lower() or \
           os.path.exists('/usr/local/share/Bayes_HEP') else 'host'


In [4]:
# Run in apptainer kernel only

if get_kernel() == 'apptainer':
    from Bayes_HEP.Design_Points import reader as Reader
    from Bayes_HEP.Design_Points import design_points as DesignPoints
    from Bayes_HEP.Design_Points import plots as Plots
    from Bayes_HEP.Design_Points import data_pred as DataPred
    from Bayes_HEP.Emulation import emulation as Emulation
    from Bayes_HEP.Calibration import calibration as Calibration
    from Bayes_HEP.Design_Points import rivet_html_parser as RivetParser

else:
    print("⚠️ Switch to Bayes HEP (Apptainer) kernel to run physics cells.")


## 2. Directory Setup

Creates the `output/` working directory where all plots, emulator files, and calibration results are stored.

If `clear_output = True`, the entire `output/` folder is deleted and recreated from scratch. This is recommended when starting a new run to avoid mixing outputs from different parameter sets. Set it to `False` if you only want to re-run later stages without repeating emulator training.

In [5]:
# Run for both kernels

if clear_output and os.path.exists(output_dir):
    print(f"Clearing output directory: {output_dir}")
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir + "/plots", exist_ok=True)

## 3. Design Points

Loads and merges the Pythia8 design point files produced by `Rivet_Notebook.ipynb`.

<details>
<summary>
More information
</summary>

**Design points** are the specific parameter combinations at which Pythia8 was run. Rather than sampling randomly, Latin Hypercube Sampling (LHS) was used in the Rivet notebook to cover the parameter space efficiently with as few runs as possible.

Each design point is one row: a set of values for the Pythia8 parameters (`pT0Ref`, `ecmPow`, `coreRadius`, `coreFraction`, `CRrange`). Multiple `Design__Rivet__<N>.dat` files are merged here into a single file so that runs from different batches are combined into one training set.

### Two outputs

- **Training set** — `train_size` design points used to fit the GP emulator
- **Validation set** — `validation_size` held-out points used to check emulator accuracy (not seen during training)

### Key variables
| Variable | Description |
|----------|-------------|
| `train_size` | Number of design points used to train the emulator |
| `validation_size` | Number of design points held out for validation |

#

> **Background:** See **slides 26–27** for a visual explanation of Latin Hypercube Sampling and the DETMAX algorithm.

</details>

In [ ]:
# Run in apptainer kernel only

input_cleaned_dir = f"{output_dir}/input_cleaned"
merged_Design_file = "Design__Rivet__Merged.dat"
merged_output_file = f'{input_cleaned_dir}/Design/{merged_Design_file}'

prediction_dir = f"{input_cleaned_dir}/Prediction"
data_dir       = f"{input_cleaned_dir}/Data"

if not os.path.exists(input_cleaned_dir):
    shutil.copytree(input_dir, input_cleaned_dir)

    if not os.path.exists(merged_output_file):
        shutil.copy(f"{input_cleaned_dir}/Rivet/parameter_prior_list.dat", merged_output_file)
        index_files   = DataPred.get_design_index(input_cleaned_dir)
        existing_rows = DataPred.get_existing_design_points(index_files)

        with open(merged_output_file, 'a') as f:
            f.write(f"\n\n# Total Design Points Merged = {len(existing_rows)}")
            f.write('\n' + "# Design point indices (row index): " + ' '.join(str(i) for i in range(len(existing_rows))) + '\n')   
            f.write("\n".join(existing_rows) + "\n")

        print(f"➕ Appended {len(existing_rows)} design points to {merged_output_file}")

        design_dir = f"{input_cleaned_dir}/Design"
        for filename in os.listdir(design_dir):
            filepath = os.path.join(design_dir, filename)
            if filepath != merged_output_file:
                os.remove(filepath)
    
    #Merging and cleaning files in input_cleaned
    DG_predictions_files = glob.glob(f"{prediction_dir}/*.dat")
    DataPred.group_histograms_by_design(DG_predictions_files, prediction_dir)
    DataPred.zeros_nan_remover(main_dir, prediction_dir, data_dir) 


In [ ]:
# Run in apptainer kernel only

RawDesign = Reader.ReadDesign(f'{input_cleaned_dir}/Design/{merged_Design_file}')
priors, parameter_names, dim = DesignPoints.get_prior(RawDesign)
train_points, validation_points, train_indices, validation_indices = DesignPoints.load_data(train_size, validation_size, RawDesign['Design'], priors, validation_indices_file=f'{input_cleaned_dir}/validation_indices.txt')

Plots.plot_design_points(output_dir, train_points, validation_points, priors)

## 4. Load Data and Predictions

Loads experimental measurements and Pythia8 predictions for each collision system, then splits the design points into training and validation sets.

<details>
<summary>
More information
</summary>

Two datasets are loaded for each collision system:
- **Experimental data** — real detector measurements (STAR, PHENIX, etc.) including statistical and systematic uncertainties, read from `input/Data/`
- **Pythia8 predictions** — simulated values at each design point, read from `input/Prediction/`

The plot written to `output/plots/` shows how the design points cover the 5-dimensional parameter space.

The 5 parameters being tuned are:

| Parameter | Description |
|-----------|-------------|
| `pT0Ref` | Reference scale for MPI regularization |
| `ecmPow` | Energy dependence of MPI |
| `coreRadius` | Size of the proton hard core |
| `coreFraction` | Fraction of matter in the core |
| `CRrange` | Range of color reconnection |

#

> **Background:** See **slides 31–35** for an overview of the experimental observables and data sources.

</details>

In [ ]:
# Run in apptainer kernel only

all_data = {}
n_hist   = {}

for system in Coll_System:

    if system == 'all':
        sys = 'all'

        prediction_files = sorted(glob.glob(os.path.join(prediction_dir, f"Prediction__{model}__*__values.dat")))
        data_files = sorted(glob.glob(os.path.join(data_dir, f"Data__*.dat")))

    else:
        System, Energy = system.split('_')[0], system.split('_')[1]  
        sys = System + Energy   

        prediction_files = sorted(glob.glob(os.path.join(prediction_dir, f"Prediction__{model}__{Energy}__{System}__*__values.dat")))
        data_files = sorted(glob.glob(os.path.join(data_dir, f"Data__{Energy}__{System}__*.dat")))

    all_predictions  = [Reader.ReadPrediction(f) for f in prediction_files]
    all_data[sys] = [Reader.ReadData(f) for f in data_files]

    n_hist[sys] = len(prediction_files)

    x, x_errors, y_data_results, y_data_errors = DataPred.get_data(all_data[sys], sys)
    y_train_results, y_train_errors, y_val_results, y_val_errors = DataPred.get_predictions(all_predictions, train_indices, validation_indices, sys)

print("Data and predictions loaded successfully.")

## 5. Gaussian Process Emulator

Trains a Gaussian Process (GP) surrogate model on the design point predictions, then validates on the held-out set.

<details>
<summary>
More information
</summary>

A **Gaussian Process** is a probabilistic model that interpolates between the design points. Given a new parameter set, the GP returns a prediction *and* an uncertainty estimate — telling us not just what Pythia8 would output, but how confident we are in that prediction.

The `validation_size` held-out points are used to compute the Root Mean Square Error (RMSE). A low validation RMSE means the emulator reliably reproduces Pythia8 outputs it hasn't seen. If the RMSE is large, the emulator may not be reliable for calibration.

### Method Selection

The emulator method is set by the `PCA` flag:

| Flag | Method | Description |
|------|--------|-------------|
| `PCA = True` | `PCGP` | Applies PCA to compress the output space, then fits independent GPs on each principal component. Better for high-dimensional output (many observables); reduces overfitting and speeds up training. |
| `PCA = False` | `indGP` | Fits one GP per observable directly in the original output space. Simpler and interpretable, but can struggle when observables are correlated or the output dimension is large. |

### Execution Modes

| Flag | Behavior |
|------|----------|
| `Submit_job_train = True` | Submits a SLURM batch job to Isaac HPC. Use for large design matrices or slow Rivet runs. |
| `Train_Surmise = True` | Trains the emulator interactively in-notebook. |
| `Load_Surmise = True` | Loads a previously saved emulator from `output/emulator/`. Skips retraining. |

Set `Load_Surmise = True` when the design points haven't changed and you want to avoid retraining between notebook runs. Retraining is only necessary when the training data, scaler, or method changes.

After training or loading, RMSE plots are written to `output/plots/emulators/` comparing train and validation residuals across all observables.

> **Background:** See **slides 36–40** for an explanation of GP regression, kernel functions, and leave-one-out validation.

</details>

In [7]:
# Run for both kernels

Emulators       = {}
PredictionVal   = {}
PredictionTrain = {}

if Submit_job_train:
    if get_kernel() == 'host':    
        for system in Coll_System:
                !sbatch --parsable --ntasks=1 --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                    {hpc_dir}/run_bayes_emulator.slurm {project} {input} {output} {system} {model} {train_size} {validation_size} {PCA} {CONTAINER} {BIND_PATH}
    else:
        print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")

elif Train_Surmise:
    print("Training Surmise emulators.")
    os.makedirs(output_dir + "/emulator", exist_ok=True)
    method_type = 'PCGP' if PCA else 'indGP'
    Emulators['surmise'], PredictionVal['surmise_val'], PredictionTrain['surmise_train'] = Emulation.train_surmise(Emulators, x, y_train_results, train_points, validation_points, output_dir, method_type)
    Plots.plot_rmspe_comparison(y_train_results, y_val_results, PredictionTrain, PredictionVal, output_dir)
    Plots.plot_rmspe_per_inspire(y_val_results, PredictionVal, all_data, x, output_dir, 'Validation')
    Plots.plot_rmspe_vs_emuvar(y_val_results, PredictionVal, all_data, x, Emulators, validation_points, output_dir, 'Validation')

elif Load_Surmise:
    print("Loading Surmise emulator.")
    Emulators['surmise'] = {}
    Emulators['surmise'], PredictionVal['surmise_val'], PredictionTrain['surmise_train'] = Emulation.load_surmise(Emulators['surmise'], x, train_points, validation_points, output_dir)

## 6. Closure Test

Validates the calibration pipeline itself by calibrating against a held-out design point with a known true answer, before trusting it against real experimental data.

<details>
<summary>
More information
</summary>

A **closure test** validates the calibration pipeline itself, independent of any bias in the real experimental data. One validation design point (`closure_index`) is temporarily treated as if it were the "data": its Pythia8-simulated values become the calibration target, and its known parameter values become the **truth**. Calibration then runs exactly as it would against real data, and the resulting posterior is compared against that known truth.

If the true parameter values fall within the recovered posterior's credible region, the pipeline (emulator + likelihood + sampler) is working correctly. If they don't, something upstream is broken — this catches problems in the emulator, likelihood, or sampler settings before you spend time calibrating against real data.

### Execution Modes

| Flag | Behavior |
|------|----------|
| `Submit_job_closure = True` | Submits a SLURM batch job to run the closure calibration on Isaac HPC. Recommended — closure calibration takes as long as a real calibration run. |
| `Run_Closure = True` | Runs the closure calibration interactively in-notebook. Use for quick tests with a small sample count only. |
| `Run_Closure_Plots = True` | Generates closure plots and the truth comparison. Run this **after** the calibration (interactive or SLURM) has finished — it only reads and plots the saved results, it does not rerun calibration. |

### Key variable

| Variable | Description |
|----------|-------------|
| `closure_index` | Index into the validation set (`0` to `validation_size - 1`) used as the pseudo-data point. |

Closure plots are written to `output/plots/closure/`.

</details>

In [ ]:
closure_name = f'closure_{closure_index}'
truths = dict(zip(parameter_names, validation_points[closure_index]))

y_pseudo = {}
for system in Coll_System:
    y_pseudo[system] = y_val_results[system][closure_index]

if Submit_job_closure:
        if get_kernel() == 'host': 
                for system in Coll_System:
                        for sampler in SAMPLERS:
                                !sbatch --parsable --ntasks={npool} --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                                        {hpc_dir}/run_bayes_closure.slurm {project} {input} {output} {system} {model} {sampler} {npool} {closure_index} {Samples} {dlog} {thin_samples} {burn_frac} {CONTAINER} {BIND_PATH} 
        else:
                print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")

elif Run_Closure:
    print("Running closure calibration.")

    sampler_configs = {
        "dynesty":         dict(nlive=200*dim, dlogz=dlog, sample='rwalk', bound='multi', facc=0.5, walks=10*dim),
        "emcee":           dict(nwalkers=200*dim, nsteps=Samples, thin=thin_samples, nburn=0, burn_in_fraction=burn_frac),
        "ultranest":       dict(nlive=100*dim, show_status=True, dlogz=dlog),
        }
    samplers = {name: sampler_configs[name] for name in SAMPLERS}

    os.makedirs(f"{output_dir}/plots/closure/", exist_ok=True)
    Calibration.run_calibration(x, y_pseudo, y_data_errors, priors, Emulators, output_dir, samplers, npool, Samples, closure_name, scalers, cov_mode)

if Run_Closure_Plots:
    print("Generating closure plots.")
    Plots.combined_results(size, x, all_data, y_pseudo, y_data_errors, Emulators, SAMPLERS, n_hist, output_dir, scalers=None, band_threshold=90, design_points=RawDesign['Design'], result_type=closure_name, legend=False)
    Plots.results(size, x, all_data, y_pseudo, y_data_errors, Emulators, SAMPLERS, n_hist, output_dir, scalers=None, band_threshold=90, design_points=RawDesign['Design'], result_type=closure_name, legend=True)
    Calibration.closure(Coll_System, Emulators, output_dir, closure_name, truths)

## 7. Bayesian Calibration

Uses Bayes' theorem to update our beliefs about the Pythia8 parameters given the experimental data.

<details>
<summary>
More information
</summary>

$$P(\theta \mid \text{data}) \propto P(\text{data} \mid \theta) \times P(\theta)$$

- $P(\theta)$ — **prior**: initial belief about the parameters (the allowed ranges)
- $P(\text{data} \mid \theta)$ — **likelihood**: how probable the data is given a particular parameter set, evaluated using the emulator
- $P(\theta \mid \text{data})$ — **posterior**: updated belief about the parameters after seeing the data

### Samplers

| Sampler | Method | Best for |
|---------|--------|----------|
| `emcee` | Ensemble MCMC — many walkers propose moves based on each other | Large, smooth posteriors |
| `dynesty` | Nested sampling — estimates Bayesian evidence and handles multimodal posteriors | Evidence estimation, complex geometry |

**Warning:** Calibration can take hours on a laptop. For production runs (10,000+ steps, 150 walkers), use `Submit_job_calibrate = True` to submit via SLURM.

> **Background:** See **slides 41–47** for an overview of Bayes' theorem, MCMC, emcee, and nested sampling.

</details>

In [8]:
# Run for both kernels

if Submit_job_calibrate:
        if get_kernel() == 'host': 
                for system in Coll_System:
                        for sampler in SAMPLERS:
                                !sbatch --parsable --ntasks={npool} --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                                        {hpc_dir}/run_bayes_calibration.slurm {project} {input} {output} {system} {model} {sampler} {npool} {Samples} {dlog} {thin_samples} {burn_frac} {CONTAINER} {BIND_PATH} 
        else:
                print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")

elif Run_Calibration:
    print("Running calibration.")

    sampler_configs = {
        "dynesty":         dict(nlive=200*dim, dlogz=dlog, sample='rwalk', bound='multi', facc=0.5, walks=10*dim),
        "emcee":           dict(nwalkers=200*dim, nsteps=Samples, thin=thin_samples, nburn=0, burn_in_fraction=burn_frac),
        "ultranest":       dict(nlive=100*dim, show_status=True, dlogz=dlog),
        }

    samplers = {name: sampler_configs[name] for name in SAMPLERS}

    os.makedirs(f"{output_dir}/plots/calibration/", exist_ok=True)
    Calibration.run_calibration(x, y_data_results, y_data_errors, priors, Emulators, output_dir, samplers, npool, Samples, 'calibration', scalers, cov_mode)

## 8. Results

Visualizes the calibration output as trace plots and posterior predictive bands.

<details>
<summary>
More information
</summary>

### Trace Plots

A trace plot shows the value of each parameter at every MCMC iteration. A well-converged chain looks like "fuzzy caterpillars" — the walkers mix freely and don't drift. We show the last `percent` fraction of the chain (after burn-in), thinned for readability, with 10 individual walkers shown in different colors.

### Posterior Predictive Plots

We draw `size` samples from the posterior and run each through the emulator to produce a distribution of model predictions:

- **Median prediction** line
- **95% credible interval** band — the range containing 95% of predictions
- **MAP** line — Maximum A Posteriori, the single best-fit parameter set

These are overlaid on the experimental data with error bars. If the calibration worked well, the data should fall within the prediction band.

Plots are written to `output/plots/results_<system>/`.

> **Background:** See **slides 48–52** for examples of corner plots, predictive bands, and MAP estimates.

</details>

In [ ]:
if Submit_job_results:
        if get_kernel() == 'host': 
                samplers_str = ",".join(SAMPLERS)
                coll_system_str = ",".join(Coll_System)
                !sbatch --parsable --ntasks={npool} --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                        {hpc_dir}/run_bayes_results.slurm {project} {input} {output} {coll_system_str} {model} {train_size} {validation_size} {samplers_str} {size} {CONTAINER} {BIND_PATH}

        else:
                print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")

elif Run_Results:
        Plots.plot_diagnostics(output_dir, x, Emulators, SAMPLERS, parameter_names)
        Calibration.merge(Coll_System, Emulators, output_dir)
        Plots.plot_uncertainty_comparison(x, all_data, y_data_results, y_data_errors, Emulators, SAMPLERS, output_dir)
        Plots.results(size, x, all_data, y_data_results, y_data_errors, y_val_results, PredictionVal, Emulators, SAMPLERS, n_hist, output_dir, scalers=None, design_points=RawDesign['Design'])  

        if show_plots:
                from IPython.display import Image, display

                for system in x.keys():
        
                        for f in sorted(glob.glob(f"{output_dir}/plots/results_{system}/*/*.png")):
                            print(f.split('/')[-1])
                            display(Image(filename=f))

## 9. Validate at the MAP Point

Re-runs the actual Pythia8 + Rivet simulation at the calibrated best-fit point, so it can be checked against the real simulator rather than just the emulator's prediction.

<details>
<summary>
More information
</summary>

Calibration in Section 7 finds the **MAP** (Maximum A Posteriori) point — the single best-fit parameter set — using the *emulator's* predictions. This section re-runs the actual Pythia8 + Rivet simulation at that MAP point, with high statistics (`nevents`), so the true simulator output can be compared directly against the experimental data rather than the emulator's approximation of it. It is a validation step, not part of calibration itself.

Unlike other stages in this notebook, MAP validation only runs via SLURM (Isaac/host kernel) — there is no interactive mode.

### Steps (run in order)

| Flag | Behavior |
|------|----------|
| `Submit_Job_MAP = True` | Runs Pythia8 + Rivet at the MAP point with `nevents` events. Uses `RUN_PT_HAT_BINS` the same way as the model run in `Rivet_Notebook.ipynb`. |
| `MAP_merge = True` | Merges the resulting `.yoda` output into a single file. Run after the `Submit_Job_MAP` job has finished. |
| `MAP_write = True` | Writes the `Data`/`Prediction` input files for the MAP point. Run after the `MAP_merge` job has finished. |

Only set one of these flags to `True` at a time — run them in sequence, waiting for each SLURM job to complete before moving to the next.

### Key variables

| Variable | Description |
|----------|-------------|
| `nevents` | Number of events to simulate at the MAP point (set in Configuration) — typically much higher than a single normal design-point run, since there's only one point to simulate. |
| `RUN_PT_HAT_BINS` | Same $\hat{p}_T$-binning behavior as in `Rivet_Notebook.ipynb`. |
| `num_MAP`, `Coll_System_MAP` | Currently hardcoded in this cell (`num_MAP = 1`, `Coll_System_MAP = ['pp_200']`) rather than set in Configuration — edit them directly here if you need a different collision system. |

</details>

In [ ]:
input_cleaned_dir = f"{output_dir}/input_cleaned"
num_MAP = 1
equiv_on = "true" 

if RUN_PT_HAT_BINS:
    PT_EDGES    = [0, 15, 20, 25, 30, 40, 60]
    pt_hat_flag = "true" 
else:
    PT_Min      = -1
    PT_Max      = -1
    pt_hat_flag = "false"

Coll_System_MAP = ['pp_200'] 
if get_kernel() == 'host':
    if Submit_Job_MAP:
    
        for system in Coll_System_MAP:

            pt_edges_arg = "-1 -1" if not RUN_PT_HAT_BINS else ' '.join(str(e) for e in PT_EDGES)
            num_tasks  = 2 if not RUN_PT_HAT_BINS else len(PT_EDGES)
            

            !sbatch --parsable --array=0-0 --ntasks=7 --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                {hpc_dir}/run_rivet_MAP.slurm {input_cleaned_dir} {pt_hat_flag} "{pt_edges_arg}" {project} {system} {num_MAP} {nevents} {CONTAINER} {BIND_PATH}
    
    elif MAP_merge:
        for system in Coll_System_MAP:
            pass_type = "merge"

            !sbatch --parsable --array=0-0 --ntasks=7 --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                    {hpc_dir}/run_rivet_mergeMAP.sh {input_cleaned_dir} {pass_type} {project} {system} {num_MAP} {equiv_on} {CONTAINER} {BIND_PATH}

    elif MAP_write: 
        for system in Coll_System_MAP:
            pass_type = "write"

            !sbatch --parsable --array=0-0 --ntasks=7 --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                    {hpc_dir}/run_rivet_mergeMAP.slurm {input_cleaned_dir} {pass_type} {project} {system} {num_MAP} {CONTAINER} {BIND_PATH}


else:
        print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")   